# Unidad II · Notebook 2: Limpieza, transformación y ETL

Fase 3 de CRISP-DM (**preparación de los datos**). Aplica las decisiones tomadas en el Notebook 1
mediante un proceso **ETL**:

- **E**xtract: leer `data/raw/student-por.csv` (sin modificarlo).
- **T**ransform: limpiar, crear variables derivadas, codificar y escalar.
- **L**oad: guardar en `data/processed` y generar el script para cargar el Data Mart en SQL Server.

Las funciones están en `src/etl.py` para que el proceso se pueda repetir con un solo comando.
Aquí se ejecutan **paso a paso** para revisar el resultado de cada una.

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.append(str(Path("../src").resolve()))
import etl

## 1. Extract: leer los datos originales

In [2]:
df = etl.extraer()
df.shape

(649, 33)

## 2. Transform – limpieza y variables derivadas (`etl.limpiar`)

| Paso | Qué hace | Por qué |
|---|---|---|
| `id_estudiante` | Agrega un identificador del 1 al 649 | El CSV no trae llave; el Data Mart la necesita |
| `abandono` | 1 si G3 = 0 | Los 15 casos tienen 0 faltas: no presentaron la evaluación |
| `aprobado` | 1 si G3 ≥ 10 | Variable objetivo de clasificación |
| `nivel_desempeno` | Bajo 0-9, Medio 10-13, Alto 14-20 | Clasificación alternativa en 3 niveles |
| `absences_recortado` | Faltas recortadas a 15 (límite IQR) | Evitar que los atípicos dominen los modelos |

In [3]:
df_limpio = etl.limpiar(df)
df_limpio[["id_estudiante", "G3", "abandono", "aprobado", "nivel_desempeno",
           "absences", "absences_recortado"]].head(10)

,id_estudiante,G3,abandono,aprobado,nivel_desempeno,absences,absences_recortado
0,1,11,0,1,Medio,4,4
1,2,11,0,1,Medio,2,2
2,3,12,0,1,Medio,6,6
3,4,14,0,1,Alto,0,0
4,5,13,0,1,Medio,0,0
5,6,13,0,1,Medio,6,6
6,7,13,0,1,Medio,0,0
7,8,13,0,1,Medio,2,2
8,9,17,0,1,Alto,0,0
9,10,13,0,1,Medio,0,0


In [4]:
resumen = pd.Series({
    "registros": len(df_limpio),
    "abandonos": df_limpio["abandono"].sum(),
    "aprobados": df_limpio["aprobado"].sum(),
    "no aprobados": (df_limpio["aprobado"] == 0).sum(),
    "faltas recortadas": (df_limpio["absences"] != df_limpio["absences_recortado"]).sum(),
})
resumen

registros            649
abandonos             15
aprobados            549
no aprobados         100
faltas recortadas     21
dtype: int64

In [5]:
df_limpio["nivel_desempeno"].value_counts()

nivel_desempeno
Medio    355
Alto     194
Bajo     100
Name: count, dtype: int64

## 3. Transform – codificación para los modelos (`etl.codificar`)

Los algoritmos de scikit-learn solo aceptan números:

- **Binarias → 0/1.** Ej. `higher`: yes = 1, no = 0; `school`: GP = 1, MS = 0.
- **Nominales → one-hot.** Ej. `Mjob` se convierte en columnas `Mjob_health`, `Mjob_services`...
  con 0 o 1. Se usa `drop_first=True` para no crear una columna redundante.
- Se **excluyen los 15 abandonos** y se usan las faltas recortadas.

In [6]:
df_modelo = etl.codificar(df_limpio)
df_modelo.shape

(634, 44)

In [7]:
df_modelo.head()

,id_estudiante,school,sex,age,address,famsize,Pstatus,Medu,Fedu,traveltime,...,Mjob_teacher,Fjob_health,Fjob_other,Fjob_services,Fjob_teacher,reason_home,reason_other,reason_reputation,guardian_mother,guardian_other
0,1,1,1,18,1,1,0,4,4,2,...,0,0,0,0,1,0,0,0,1,0
1,2,1,1,17,1,1,1,1,1,1,...,0,0,1,0,0,0,0,0,0,0
2,3,1,1,15,1,0,1,1,1,1,...,0,0,1,0,0,0,1,0,1,0
3,4,1,1,15,1,1,1,4,2,1,...,0,0,0,1,0,1,0,0,1,0
4,5,1,1,16,1,1,1,3,3,1,...,0,0,1,0,0,1,0,0,0,0


Resultado: **634 registros** (649 − 15 abandonos) y todas las columnas numéricas.

## 4. Transform – escalamiento (`etl.escalar`)

`StandardScaler` deja cada variable con **media 0 y desviación 1**. Sin esto, `absences` (0-15)
pesaría más que `studytime` (1-4) en algoritmos basados en distancias como **K-Means** (Unidad IV).

> En la Unidad III el escalamiento se hará **dentro del pipeline**, ajustado solo con los datos de
> entrenamiento, para no filtrar información del conjunto de prueba.

In [8]:
df_escalado = etl.escalar(df_modelo)
df_escalado[["studytime", "absences", "failures", "Medu"]].describe().round(2)

,studytime,absences,failures,Medu
count,634.00,634.00,634.00,634.00
mean,0.00,0.00,-0.00,0.00
std,1.00,1.00,1.00,1.00
min,-1.13,-0.88,-0.36,-2.23
25%,-1.13,-0.88,-0.36,-0.46
50%,0.07,-0.39,-0.36,-0.46
75%,0.07,0.59,-0.36,1.31
max,2.47,2.79,4.82,1.31


## 5. Diseño del Data Mart (modelo estrella)

```
                 dim_estudiante
                       |
  dim_familia ---- fact_rendimiento ---- dim_escuela
                    /          \
           dim_habitos      dim_desempeno
```

- **Tabla de hechos `fact_rendimiento`:** lo que se mide → `g1`, `g2`, `g3`, `faltas`,
  `reprobadas_previas`, `aprobado`, `abandono`.
- **Dimensiones:** lo que describe a los hechos:
  - `dim_estudiante`: sexo, edad, domicilio, aspiración a estudios superiores...
  - `dim_familia`: escolaridad y trabajo de los padres, tutor, relación familiar...
  - `dim_habitos`: tiempo de estudio, salidas, consumo de alcohol, apoyos...
  - `dim_escuela`: GP / MS.
  - `dim_desempeno`: Bajo / Medio / Alto con sus rangos de calificación.

Los valores se traducen al español (ej. `yes` → `Sí`, `at_home` → `Hogar`) para que el dashboard
de Power BI sea legible.

In [9]:
tablas = etl.tablas_datamart(df_limpio)
pd.Series({nombre: len(t) for nombre, t in tablas.items()})

dim_escuela           2
dim_desempeno         3
dim_estudiante      649
dim_familia         649
dim_habitos         649
fact_rendimiento    649
dtype: int64

In [10]:
tablas["dim_familia"].head()

,id_familia,tamano_familia,estado_padres,escolaridad_madre,escolaridad_padre,trabajo_madre,trabajo_padre,tutor,apoyo_familiar,relacion_familiar
0,1,Más de 3,Separados,Superior,Superior,Hogar,Docente,Madre,No,4
1,2,Más de 3,Juntos,Primaria (4°),Primaria (4°),Hogar,Otro,Padre,Sí,5
2,3,3 o menos,Juntos,Primaria (4°),Primaria (4°),Hogar,Otro,Madre,No,4
3,4,Más de 3,Juntos,Superior,5° a 9°,Salud,Servicios,Madre,Sí,3
4,5,Más de 3,Juntos,Secundaria,Secundaria,Otro,Otro,Padre,Sí,4


In [11]:
tablas["fact_rendimiento"].head()

,id_hecho,id_estudiante,id_escuela,id_familia,id_habitos,id_desempeno,g1,g2,g3,faltas,reprobadas_previas,aprobado,abandono
0,1,1,1,1,1,2,0,11,11,4,0,1,0
1,2,2,1,2,2,2,9,11,11,2,0,1,0
2,3,3,1,3,3,2,12,13,12,6,0,1,0
3,4,4,1,4,4,3,14,14,14,0,0,1,0
4,5,5,1,5,5,2,11,13,13,0,0,1,0


## 6. Load: guardar resultados

Se generan:

| Archivo | Contenido |
|---|---|
| `data/processed/student_por_limpio.csv` | 649 registros limpios con variables derivadas |
| `data/processed/student_por_modelo.csv` | 634 registros codificados (Unidad III) |
| `data/processed/student_por_escalado.csv` | 634 registros escalados (Unidad IV) |
| `data/processed/datamart/*.csv` | Una tabla CSV por cada tabla del Data Mart |
| `sql/02_cargar_datos.sql` | Script de `INSERT` para cargar el Data Mart en SQL Server |

In [12]:
etl.cargar(df_limpio, df_modelo, df_escalado, tablas)
sorted(p.name for p in Path("../data/processed").rglob("*.csv"))

['dim_desempeno.csv',
 'dim_escuela.csv',
 'dim_estudiante.csv',
 'dim_familia.csv',
 'dim_habitos.csv',
 'fact_rendimiento.csv',
 'student_por_escalado.csv',
 'student_por_limpio.csv',
 'student_por_modelo.csv']

## 7. Verificación

Comprobamos que el original no cambió y que los archivos procesados se leen correctamente.

In [13]:
original = pd.read_csv("../data/raw/student-por.csv", sep=";")
procesado = pd.read_csv("../data/processed/student_por_modelo.csv")
pd.Series({
    "original sin cambios (649 x 33)": original.shape == (649, 33),
    "modelo sin nulos": procesado.isna().sum().sum() == 0,
    "modelo sin abandonos": (procesado["G3"] > 0).all(),
    "modelo solo numérico": procesado.select_dtypes(exclude="number").empty,
})

original sin cambios (649 x 33)    True
modelo sin nulos                   True
modelo sin abandonos               True
modelo solo numérico               True
dtype: bool

## 8. Cargar el Data Mart en SQL Server

En **SSMS**, conectada a `localhost\SQLEXPRESS`, abrir y ejecutar (F5) en este orden:

1. `sql/01_crear_datamart.sql` → crea la base `DM_RendimientoEstudiantil` y las 6 tablas.
2. `sql/02_cargar_datos.sql` → inserta los datos.
3. `sql/03_consultas_analiticas.sql` → consultas de prueba (conteos, promedios por escuela, etc.).

## Conclusiones

- Los datos quedaron limpios, codificados y listos para la Unidad III (regresión y clasificación)
  y la Unidad IV (clustering y PCA).
- El Data Mart en modelo estrella permite consultar el rendimiento por escuela, familia, hábitos y nivel
  de desempeño, y será la fuente del dashboard de Power BI (Unidad V).
- Todo el proceso es reproducible con `python src/etl.py`.